# 🚦 Traffic Violation Detection - Custom YOLOv8 Training

Train a custom YOLOv8 model to detect:
- ✅ Helmet / ❌ No-Helmet
- 🏍️ Motorcycles, 🚗 Cars, 🚚 Trucks, 🚌 Buses
- 🚦 Traffic Lights (Red/Yellow/Green)
- 📋 License Plates

---

## Step 1: Check GPU Availability

In [ ]:
!nvidia-smi

## Step 2: Install Required Packages

In [ ]:
!pip install -q ultralytics roboflow

from IPython import display
display.clear_output()

print("✅ Packages installed successfully!")

## Step 3: Mount Google Drive (to save your model)

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

# Create output directory
output_dir = '/content/drive/MyDrive/traffic-violation-model'
os.makedirs(output_dir, exist_ok=True)

print(f"✅ Google Drive mounted!")
print(f"📁 Models will be saved to: {output_dir}")

## Step 4: Download Dataset

### Option A: Use Roboflow Dataset (Recommended)

Get a free API key from: https://roboflow.com

Search for these datasets:
- "Helmet Detection"
- "Traffic Light Detection"
- "License Plate Detection"

Or use a combined dataset if available.

In [ ]:
# METHOD 1: Roboflow API
from roboflow import Roboflow

# Replace with your API key
rf = Roboflow(api_key="YOUR_ROBOFLOW_API_KEY")

# Example: Download helmet detection dataset
# Replace with your actual workspace/project
project = rf.workspace("YOUR_WORKSPACE").project("YOUR_PROJECT")
dataset = project.version(1).download("yolov8")

dataset_path = dataset.location
print(f"✅ Dataset downloaded to: {dataset_path}")

### Option B: Upload Your Own Dataset

Upload a ZIP file with this structure:
```
dataset/
├── train/
│   ├── images/
│   └── labels/
├── valid/
│   ├── images/
│   └── labels/
└── data.yaml
```

In [ ]:
# METHOD 2: Upload ZIP file
# Uncomment if using this method

# from google.colab import files
# import zipfile

# print("📤 Upload your dataset ZIP file...")
# uploaded = files.upload()

# zip_filename = list(uploaded.keys())[0]
# with zipfile.ZipFile(zip_filename, 'r') as zip_ref:
#     zip_ref.extractall('/content/dataset')

# dataset_path = '/content/dataset'
# print(f"✅ Dataset extracted to: {dataset_path}")

## Step 5: Create/Verify data.yaml

In [ ]:
# Create data.yaml file
data_yaml_content = f'''path: {dataset_path}
train: train/images
val: valid/images

nc: 10
names:
  0: Helmet
  1: No-Helmet
  2: Motorcycle
  3: Car
  4: Truck
  5: Bus
  6: Traffic-Light-Red
  7: Traffic-Light-Green
  8: Traffic-Light-Yellow
  9: Numberplate
'''

with open('/content/data.yaml', 'w') as f:
    f.write(data_yaml_content)

print("✅ data.yaml created!")
print("\n📄 Contents:")
print(data_yaml_content)

## Step 6: Load YOLOv8 Model

In [ ]:
from ultralytics import YOLO

# Choose model size:
# yolov8s.pt - Small (fast, good for testing)
# yolov8m.pt - Medium (balanced)
# yolov8l.pt - Large (best accuracy, slower)

model = YOLO('yolov8s.pt')

print("✅ YOLOv8s model loaded!")

## Step 7: Train the Model 🚀

This will take 1-2 hours depending on dataset size and epochs.

In [ ]:
# Training configuration
results = model.train(
    data='/content/data.yaml',
    epochs=100,              # Number of training epochs
    imgsz=640,              # Image size
    batch=16,               # Batch size (reduce if GPU memory error)
    name='traffic-violation-v1',
    patience=20,            # Early stopping patience
    save=True,
    device=0,               # GPU device
    workers=8,
    optimizer='AdamW',
    lr0=0.01,              # Initial learning rate
    weight_decay=0.0005,
    augment=True,          # Data augmentation
    plots=True,            # Save plots
    verbose=True
)

print("\n" + "="*60)
print("✅ Training Complete!")
print("="*60)

## Step 8: Validate the Model

In [ ]:
# Validate on test set
metrics = model.val()

print("\n📊 Model Performance:")
print(f"  mAP50: {metrics.box.map50:.3f}")
print(f"  mAP50-95: {metrics.box.map:.3f}")
print(f"  Precision: {metrics.box.mp:.3f}")
print(f"  Recall: {metrics.box.mr:.3f}")

## Step 9: Test Predictions

In [ ]:
# Test on sample images from validation set
import glob
from IPython.display import Image, display

test_images = glob.glob(f'{dataset_path}/valid/images/*.jpg')[:5]

for img_path in test_images:
    results = model.predict(img_path, conf=0.25, save=True)
    print(f"\n📸 Predictions for: {img_path.split('/')[-1]}")

# Display first result
if test_images:
    result_path = 'runs/detect/predict/image0.jpg'
    if os.path.exists(result_path):
        display(Image(filename=result_path, width=600))

## Step 10: Save Model to Google Drive

In [ ]:
import shutil

# Path to best model
best_model_path = 'runs/detect/traffic-violation-v1/weights/best.pt'

# Copy to Google Drive
shutil.copy(best_model_path, f'{output_dir}/best.pt')

print("\n" + "="*60)
print("✅ MODEL SAVED SUCCESSFULLY!")
print("="*60)
print(f"\n📁 Location: {output_dir}/best.pt")
print("\n📥 Next Steps:")
print("  1. Download 'best.pt' from Google Drive")
print("  2. Place in: C:\\Users\\svlak\\New folder (14)\\backend\\best.pt")
print("  3. Restart your backend server")
print("  4. Test with traffic violation images!")
print("="*60)

## Step 11: Export to ONNX (Optional - Faster Inference)

In [ ]:
# Export to ONNX format for faster inference
model.export(format='onnx')

print("✅ Model exported to ONNX format!")

## 📊 View Training Results

In [ ]:
# Display training plots
from IPython.display import Image, display

plots_dir = 'runs/detect/traffic-violation-v1'

print("📈 Training Results:\n")

# Results plot
if os.path.exists(f'{plots_dir}/results.png'):
    print("Loss and Metrics:")
    display(Image(filename=f'{plots_dir}/results.png', width=800))

# Confusion matrix
if os.path.exists(f'{plots_dir}/confusion_matrix.png'):
    print("\nConfusion Matrix:")
    display(Image(filename=f'{plots_dir}/confusion_matrix.png', width=800))

---

## 🎉 Training Complete!

Your custom traffic violation detection model is ready!

**Remember to:**
1. Download `best.pt` from Google Drive
2. Place it in your backend folder
3. Restart the backend server
4. Test with real traffic images!

---